# Public Data Access

Explore public nmrXiv projects, samples, and datasets. These endpoints do not require login.

In [20]:
import os
from pprint import pprint

import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv()

BASE_URL = os.getenv("NMRXIV_BASE_URL", "https://nmrxiv.org").rstrip("/")
API_BASE = f"{BASE_URL}/api"

session = requests.Session()
session.headers.update({"Accept": "application/json", "Content-Type": "application/json"})

In [21]:
def api_request(method, path, **kwargs):
    url = f"{API_BASE}{path}"
    response = session.request(method, url, timeout=30, **kwargs)
    print(f"{response.request.method} {response.url} -> {response.status_code}")
    try:
        payload = response.json()
    except ValueError:
        payload = response.text
    if not response.ok:
        pprint(payload)
        response.raise_for_status()
    return payload


def as_dataframe(payload):
    if isinstance(payload, list):
        return pd.json_normalize(payload)
    if isinstance(payload, dict):
        for key in ["data", "projects", "samples", "datasets", "items", "results"]:
            if isinstance(payload.get(key), list):
                return pd.json_normalize(payload[key])
        return pd.json_normalize(payload)
    return pd.DataFrame({"value": [payload]})


def clean_params(params):
    return {key: value for key, value in params.items() if value not in (None, "")}


def list_models(
    model,
    per_page=100,
    page=1,
    sort="-created_at",
    name=None,
    identifier=None,
    doi=None,
    created_at=None,
):
    if model not in {"projects", "samples", "datasets"}:
        raise ValueError("model must be one of: projects, samples, datasets")
    params = clean_params({
        "per_page": per_page,
        "page": page,
        "sort": sort,
        "filter[name]": name,
        "filter[identifier]": identifier,
        "filter[doi]": doi,
        "filter[created_at]": created_at,
    })
    return api_request("GET", f"/v1/list/{model}", params=params)


def get_model(public_id):
    return api_request("GET", f"/v1/{public_id}")

## Retrieve public scientific data collections

Fetches paginated collections of publicly available scientific data models from the nmrXiv repository. Supports projects, samples, and datasets.

Documented query parameters:

- `per_page`: number of results per page, default `100`, maximum `500`
- `page`: page number for pagination, default `1`
- `sort`: one of `created_at`, `-created_at`, `identifier`, `-identifier`
- `filter[name]`: case-insensitive partial match on name/title
- `filter[identifier]`: exact NMRXIV identifier match, e.g. `P123`
- `filter[doi]`: DOI match
- `filter[created_at]`: ISO date or date range, e.g. `2024-01-01,2024-12-31`

In [22]:
projects = list_models(
    model="projects",
    per_page=100,
    page=1,
    sort="-created_at",
)
projects_df = as_dataframe(projects)
projects_df.shape

GET https://nmrxiv.org/api/v1/list/projects?per_page=100&page=1&sort=-created_at -> 200


(100, 39)

## Retrieve data by project identifier.

Uncomment or edit values to combine filters. Query parameter names are mapped by the `list_models(...)` helper.

In [23]:
filtered_projects = list_models(
    model="projects",
    per_page=20,
    page=1,
    sort="identifier",
    name="Pulegon: Die Polei-minze Im Wandel Der Zeiten",
    identifier=None,
    doi=None,
    created_at=None,
)

filtered_projects_df = as_dataframe(filtered_projects)
filtered_projects_df.head()

GET https://nmrxiv.org/api/v1/list/projects?per_page=20&page=1&sort=identifier&filter%5Bname%5D=Pulegon%3A+Die+Polei-minze+Im+Wandel+Der+Zeiten -> 200


,id,name,slug,description,photo_url,is_public,is_published,identifier,schema_version,public_url,...,owner.profile_photo_url,stats.likes,license.title,license.slug,license.spdx_id,license.url,license.html_url,license.permissions,license.description,license.body
0,30,Pulegon: Die Polei-Minze im Wandel der Zeiten,pulegon-die-polei-minze-im-wandel-der-zeiten,NMR dataset of (+)-Pulegon isolated from Menth...,https://s3.uni-jena.de/nmrxiv-public/projects/...,True,True,NMRXIV:P16,beta,https://nmrxiv.org/project/P16,...,https://s3.uni-jena.de/nmrxiv-public/profile-p...,2,Creative Commons Attribution 4.0 International...,cc-by-4.0,CC-BY-4.0,https://creativecommons.org/licenses/by/4.0/le...,None,None,You are free to: <br><b>Share</b> — copy and r...,<b>Creative Commons License Deed</b><br><b>You...


In [ ]:
# The below filtering technique is still in development and will be uncommented once deployed to production.
# project_by_identifier = list_models(
#     model="projects",
#     per_page=10,
#     page=1,
#     sort="",
#     identifier="P16",
# )

# as_dataframe(project_by_identifier).head()

In [ ]:
# The below filtering technique is still in development and will be uncommented once deployed to production.
# projects_created_in_range = list_models(
#     model="projects",
#     per_page=20,
#     page=1,
#     sort="-created_at",
#     created_at="2024-01-01,2026-12-31",
# )

# as_dataframe(projects_created_in_range).head()

In [ ]:
#projects_df.head()

## List samples and datasets

In [24]:
samples_df = as_dataframe(list_models("samples"))
datasets_df = as_dataframe(list_models("datasets"))

samples_df.shape, datasets_df.shape

GET https://nmrxiv.org/api/v1/list/samples?per_page=100&page=1&sort=-created_at -> 200
GET https://nmrxiv.org/api/v1/list/datasets?per_page=100&page=1&sort=-created_at -> 200


((100, 29), (100, 24))

## Retrieve specific public scientific data by identifier​

Fetches detailed information for a specific publicly available scientific data entry using its NMRXIV identifier. Returns comprehensive metadata, associated files, measurement details, and structured data compliant with scientific data standards. Supports projects (P prefix), samples (S prefix), and datasets (D prefix). Use identifiers such as `P16`, `S70`, or `D399`.

In [25]:
public_id = "P16"
model_detail = get_model(public_id)
model_detail

GET https://nmrxiv.org/api/v1/P16 -> 200


{'data': {'id': 30,
  'name': 'Pulegon: Die Polei-Minze im Wandel der Zeiten',
  'slug': 'pulegon-die-polei-minze-im-wandel-der-zeiten',
  'description': "NMR dataset of (+)-Pulegon isolated from Mentha pulegium, isolated from the essential oil of Mentha pulegium (Pennyroyal) obtained from OMIKRON, Rietberg, Germany. \n\nGeographic location of data collection: Leipzig, Germany; Tübingen, Germany\n\nLanguage of corresponding scientific publication: German\n\nNote: All EXPNOs were opened with TopSpin 4.3 before uploading to nmrXiv in order to save the processed spectra via TopSpin's autoprocessing feature in the PROCNOs in the pdata folder, which are processed by NMRium. The NMR data in the complete datasets in RADAR4Chem (see citation), which also contain data other than NMR data, only contain the processing parameters but not the processed spectra.",
  'team': {'id': 21,
   'user_id': 18,
   'name': 'IPB Halle',
   'personal_team': False,
   'created_at': '2022-10-04T08:50:34.000000Z',

## Helpful columns

In [26]:
columns = [col for col in ["identifier", "name", "slug", "description", "is_public", "is_published"] if col in projects_df.columns]
projects_df[columns].head(20)

,identifier,name,slug,description,is_public,is_published
0,NMRXIV:P179,BC2 Lab 2025 Class NMR (5a78b7d0 - for Biochem...,bc2-lab-2025-class-nmr-5a78b7d0-for-biochemist...,Spring 2025 Class NMR data lipase-catalyzed ki...,True,True
1,NMRXIV:P176,S-Aminodibenzo[b]thiophenium Salts as Versatil...,s-aminodibenzobthiophenium-salts-as-versatile-...,"S-Aminodibenzo[b]thiophenium salts, accessible...",True,True
2,NMRXIV:P172,250_DAMCAFF,250-damcaff,Raw analytical data associated with the articl...,True,True
3,NMRXIV:P171,NMR Chemical Shifts of Common Flavonoids,nmr-chemical-shifts-of-common-flavonoids,We present 1H and 13C nuclear magnetic resonan...,True,True
4,NMRXIV:P170,"Primary NMR data for ""Synthesis of Densely Sub...",primary-nmr-data-for-synthesis-of-densely-subs...,This project contains primary NMR fid files fo...,True,True
5,NMRXIV:P169,"Synthesis of Synthesis of desulfo ethylGSL, de...",synthesis-of-synthesis-of-desulfo-ethylgsl-des...,1H and 13C NMRs of desulfoGSLs and GSLs and th...,True,True
6,NMRXIV:P174,SYNTHESIS OF DESULFO N-HEXYLGSL AND DESULFO 5-...,synthesis-of-desulfo-n-hexylgsl-and-desulfo-5-...,1H and 13C NMRs of desulfoGSLs and GSLs and th...,True,True
7,NMRXIV:P173,305_HPJSRDS,305-hpjsrds,Study of the high congruence of the coupling c...,True,True
8,NMRXIV:P167,PHYTOCHEMICAL PROFILING AND MULTI-TARGET PHARM...,phytochemical-profiling-and-multi-target-pharm...,NMR analysis of isolated compounds from Sympho...,True,True
9,NMRXIV:P166,NMR data of Lichenic acids from Irish Cladonia...,nmr-data-of-lichenic-acids-from-irish-cladonia...,"Cladonia portentosa, commonly known as reindee...",True,True
